# 01 — Coleta e Preparação dos Dados

**Projeto:** Entregas Atrasadas e Satisfação do Cliente (Olist)

**Objetivo deste notebook:** extrair as 9 tabelas do Olist que estão no PostgreSQL (carregadas via DBeaver), aplicar apenas a preparação estrutural necessária e validar a integridade antes de seguir para a limpeza.

**O que este notebook faz**
1. Conecta ao PostgreSQL (credenciais via variáveis de ambiente, nunca no código)
2. Confere se as 9 tabelas existem e carrega cada uma em um DataFrame
3. Preparação estrutural: datas → `datetime`, CEPs → texto com 5 dígitos, remoção de espaços em texto
4. Validações: contagem de linhas, chaves primárias, integridade referencial, nulos e duplicatas
5. Salva as tabelas preparadas em `data/processed/` (Parquet)

**O que este notebook NÃO faz:** decisões de limpeza (remover nulos, duplicatas, outliers, filtrar status). Isso fica para o `02_limpeza.ipynb`. Aqui só **diagnosticamos**.

## 1. Configuração

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import URL

# Opcional: lê as credenciais de um arquivo .env (pip install python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DB = {
    "host":     os.getenv("PGHOST", "localhost"),
    "port":     int(os.getenv("PGPORT", 5432)),
    "database": os.getenv("PGDATABASE", "olist"),
    "user":     os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
}
SCHEMA = os.getenv("PGSCHEMA", "public")

# Pasta de saída (o notebook roda de dentro de notebooks/)
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# nome curto usado no projeto -> nome da tabela no PostgreSQL
# AJUSTE aqui se no DBeaver você deu outros nomes às tabelas
TABELAS = {
    "orders":               "olist_orders_dataset",
    "order_items":          "olist_order_items_dataset",
    "order_payments":       "olist_order_payments_dataset",
    "order_reviews":        "olist_order_reviews_dataset",
    "customers":            "olist_customers_dataset",
    "sellers":              "olist_sellers_dataset",
    "products":             "olist_products_dataset",
    "geolocation":          "olist_geolocation_dataset",
    "category_translation": "product_category_name_translation",
}

# Linhas esperadas segundo o dataset original do Kaggle (para conferir a carga)
ESPERADO = {
    "orders": 99441, "order_items": 112650, "order_payments": 103886,
    "order_reviews": 99224, "customers": 99441, "sellers": 3095,
    "products": 32951, "geolocation": 1000163, "category_translation": 71,
}

## 2. Conexão com o PostgreSQL

In [ ]:
url = URL.create(
    "postgresql+psycopg2",
    username=DB["user"], password=DB["password"],
    host=DB["host"], port=DB["port"], database=DB["database"],
)
engine = create_engine(url)

with engine.connect() as conn:
    versao = conn.execute(text("SELECT version()")).scalar()
print("✅ Conectado:", versao.split(",")[0])

# Confere se as 9 tabelas existem no schema
existentes = set(inspect(engine).get_table_names(schema=SCHEMA))
faltando = {k: v for k, v in TABELAS.items() if v not in existentes}
if faltando:
    raise ValueError(
        f"Tabelas não encontradas no schema '{SCHEMA}': {faltando}\n"
        f"Tabelas existentes: {sorted(existentes)}\n"
        "Ajuste o dicionário TABELAS na célula de configuração."
    )
print(f"✅ As {len(TABELAS)} tabelas foram encontradas no schema '{SCHEMA}'")

## 3. Carga das tabelas

In [ ]:
dfs = {}
for nome, tabela in TABELAS.items():
    dfs[nome] = pd.read_sql_query(text(f'SELECT * FROM "{SCHEMA}"."{tabela}"'), engine)
    print(f"{nome:22s} {dfs[nome].shape[0]:>10,} linhas × {dfs[nome].shape[1]:>2} colunas")

### Pré-visualização rápida
Primeira olhada na tabela central (`orders`) e nos tipos de dados que vieram do banco.

In [ ]:
display(dfs["orders"].head())
dfs["orders"].dtypes

## 4. Preparação estrutural

Ajustes **que não alteram o conteúdo** dos dados, só corrigem o tipo/formato:

- **Datas** vêm como texto → convertidas para `datetime` (valores inválidos viram `NaT`, e contamos quantos)
- **CEPs** (`zip_code_prefix`) podem ter perdido o zero à esquerda se o DBeaver os importou como número → padronizados com 5 dígitos em texto
- **Espaços** no início/fim de textos removidos
- **Numéricos** garantidos como número

> Os nomes de colunas originais foram mantidos (inclusive `product_name_lenght` e `product_description_lenght`, que vêm com erro de grafia no dataset original). Qualquer renomeação fica para a etapa de limpeza.

In [ ]:
DATE_COLS = {
    "orders": [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items":   ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}
ZIP_COLS = {
    "customers":   ["customer_zip_code_prefix"],
    "sellers":     ["seller_zip_code_prefix"],
    "geolocation": ["geolocation_zip_code_prefix"],
}
NUM_COLS = {
    "order_items":    ["order_item_id", "price", "freight_value"],
    "order_payments": ["payment_sequential", "payment_installments", "payment_value"],
    "order_reviews":  ["review_score"],
    "products": [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    ],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}

log = []  # registro do que foi feito, para documentar no README

for nome, df in dfs.items():
    # 1) remove espaços em colunas de texto
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].str.strip()

    # 2) datas -> datetime
    for c in DATE_COLS.get(nome, []):
        antes = df[c].notna().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce")
        invalidos = antes - df[c].notna().sum()
        log.append((nome, c, "→ datetime", f"{invalidos} valores inválidos viraram NaT"))

    # 3) CEP -> texto de 5 dígitos
    for c in ZIP_COLS.get(nome, []):
        df[c] = (df[c].astype("string").str.strip()
                      .str.replace(r"\.0$", "", regex=True).str.zfill(5))
        log.append((nome, c, "→ string(5)", "zeros à esquerda restaurados"))

    # 4) numéricos
    for c in NUM_COLS.get(nome, []):
        df[c] = pd.to_numeric(df[c], errors="coerce")
        log.append((nome, c, "→ numérico", ""))

    dfs[nome] = df

pd.DataFrame(log, columns=["tabela", "coluna", "ajuste", "observação"])

## 5. Validação da carga

### 5.1 Contagem de linhas vs. dataset original
Se algum valor divergir, a carga no DBeaver pode ter perdido linhas (encoding, delimitador, aspas em comentários etc.).

In [ ]:
resumo = pd.DataFrame({
    "tabela":   list(dfs),
    "linhas":   [len(d) for d in dfs.values()],
    "colunas":  [d.shape[1] for d in dfs.values()],
    "esperado": [ESPERADO[k] for k in dfs],
    "memoria_mb": [round(d.memory_usage(deep=True).sum() / 1e6, 1) for d in dfs.values()],
})
resumo["diferenca"] = resumo["linhas"] - resumo["esperado"]
resumo["ok"] = resumo["diferenca"].eq(0).map({True: "✅", False: "⚠️"})
resumo

### 5.2 Chaves primárias
Cada tabela deveria ter uma chave única. Duplicidades aqui são **diagnóstico**, ainda não corrigimos nada.
(`order_reviews` e `geolocation` não têm chave única garantida no dataset original.)

In [ ]:
PKS = {
    "orders":               ["order_id"],
    "customers":            ["customer_id"],
    "sellers":              ["seller_id"],
    "products":             ["product_id"],
    "order_items":          ["order_id", "order_item_id"],
    "order_payments":       ["order_id", "payment_sequential"],
    "category_translation": ["product_category_name"],
    "order_reviews":        ["review_id", "order_id"],
}

linhas = []
for nome, chave in PKS.items():
    df = dfs[nome]
    linhas.append({
        "tabela": nome,
        "chave": " + ".join(chave),
        "chaves_duplicadas": int(df.duplicated(subset=chave).sum()),
        "chaves_nulas": int(df[chave].isna().any(axis=1).sum()),
    })
pd.DataFrame(linhas)

### 5.3 Integridade referencial
Verifica se as chaves estrangeiras apontam para registros que existem (ex.: todo `order_id` em `order_items` existe em `orders`?). Valores nulos são ignorados.

In [ ]:
FKS = [
    ("orders",         "customer_id",           "customers",            "customer_id"),
    ("order_items",    "order_id",              "orders",               "order_id"),
    ("order_items",    "product_id",            "products",             "product_id"),
    ("order_items",    "seller_id",             "sellers",              "seller_id"),
    ("order_payments", "order_id",              "orders",               "order_id"),
    ("order_reviews",  "order_id",              "orders",               "order_id"),
    ("products",       "product_category_name", "category_translation", "product_category_name"),
]

linhas = []
for filha, col_f, mae, col_m in FKS:
    valores = dfs[filha][col_f].dropna()
    orfaos = ~valores.isin(dfs[mae][col_m])
    linhas.append({
        "relação": f"{filha}.{col_f} → {mae}.{col_m}",
        "registros_órfãos": int(orfaos.sum()),
        "valores_distintos_órfãos": int(valores[orfaos].nunique()),
        "status": "✅" if orfaos.sum() == 0 else "⚠️",
    })
pd.DataFrame(linhas)

### 5.4 Valores nulos
Só colunas com pelo menos um nulo. Nulos em `order_delivered_customer_date` são esperados (pedidos não entregues); os demais serão avaliados na limpeza.

In [ ]:
nulos = []
for nome, df in dfs.items():
    n = df.isna().sum()
    n = n[n > 0]
    for col, qtd in n.items():
        nulos.append({"tabela": nome, "coluna": col, "nulos": int(qtd),
                      "pct": round(100 * qtd / len(df), 2)})

tabela_nulos = pd.DataFrame(nulos).sort_values("pct", ascending=False).reset_index(drop=True)
tabela_nulos

### 5.5 Linhas totalmente duplicadas

In [ ]:
dups = pd.DataFrame({
    "tabela": list(dfs),
    "linhas_duplicadas": [int(d.duplicated().sum()) for d in dfs.values()],
})
dups["pct"] = (100 * dups["linhas_duplicadas"] / resumo["linhas"]).round(2)
dups

### 5.6 Sanidade das datas e do período
Confere o intervalo coberto e se há inconsistências lógicas óbvias na sequência do pedido.

In [ ]:
o = dfs["orders"]
print("Período das compras:", o["order_purchase_timestamp"].min().date(),
      "→", o["order_purchase_timestamp"].max().date())
print("\nDistribuição de status:")
display(o["order_status"].value_counts(dropna=False).to_frame("pedidos")
        .assign(pct=lambda d: (100 * d["pedidos"] / len(o)).round(2)))

entrega_antes_compra = (o["order_delivered_customer_date"] < o["order_purchase_timestamp"]).sum()
print(f"\nEntregas anteriores à data de compra (inconsistência): {entrega_antes_compra}")

## 6. Salvando os dados preparados

Formato **Parquet**: mantém os tipos (datetime, texto de CEP) e é bem menor que CSV. Os notebooks seguintes leem direto daqui, sem precisar do banco.

In [ ]:
for nome, df in dfs.items():
    df.to_parquet(PROCESSED / f"{nome}.parquet", index=False)

print("Arquivos salvos em", PROCESSED.resolve())
for p in sorted(PROCESSED.glob("*.parquet")):
    print(f"  {p.name:35s} {p.stat().st_size / 1e6:6.1f} MB")

## 7. Resumo para o README
Tabela em Markdown pronta para colar na seção *Step 1* do README.

In [ ]:
try:
    print(resumo[["tabela", "linhas", "colunas", "esperado", "ok"]].to_markdown(index=False))
except ImportError:
    print("Instale 'tabulate' (pip install tabulate) para gerar a tabela em Markdown.")
    print(resumo[["tabela", "linhas", "colunas", "esperado", "ok"]])

## 8. Conclusões da etapa de coleta

*Preencha depois de rodar o notebook com o que as validações mostraram. Exemplos do que registrar:*

- As 9 tabelas foram carregadas e as contagens batem com o dataset original? (seção 5.1)
- Há chaves duplicadas em `order_reviews` ou em outras tabelas? (seção 5.2)
- Existem registros órfãos, por exemplo categorias sem tradução? (seção 5.3)
- Quais colunas têm mais nulos e o que isso significa para a análise de atraso? (seção 5.4)

**Próximo passo:** `02_limpeza.ipynb` — tratar nulos, duplicatas, status de pedido e outliers, e definir a flag de atraso.